In [1]:
import pandas as pd

sessions_data = pd.read_csv("sessions_data.csv")

web_data = pd.read_csv("cleaned_web_data.csv")
web_data[["client_id", "visit_id", "process_step", "date_time", "variation"]].head(10)

,client_id,visit_id,process_step,date_time,variation
0,555,637149525_38041617439_716659,confirm,2017-04-15 13:00:34,Test
1,555,637149525_38041617439_716659,step_3,2017-04-15 13:00:14,Test
2,555,637149525_38041617439_716659,step_2,2017-04-15 12:58:35,Test
3,555,637149525_38041617439_716659,step_1,2017-04-15 12:58:03,Test
4,555,637149525_38041617439_716659,start,2017-04-15 12:57:56,Test
5,647,40369564_40101682850_311847,confirm,2017-04-12 15:47:45,Test
6,647,40369564_40101682850_311847,step_3,2017-04-12 15:45:02,Test
7,647,40369564_40101682850_311847,step_2,2017-04-12 15:41:53,Test
8,647,40369564_40101682850_311847,step_1,2017-04-12 15:41:35,Test
9,647,40369564_40101682850_311847,start,2017-04-12 15:41:28,Test


In [2]:
# We check the data type of date_time & correct it
web_data["date_time"].dtype

<StringDtype(na_value=nan)>

In [3]:
web_data["date_time"] = pd.to_datetime(web_data["date_time"])
web_data["date_time"].dtype

dtype('<M8[us]')

In [4]:
# We calculate the duration of every step
# First, we group the records by client and visit with the earliest timestamp first

timed_steps = (
    web_data
    .sort_values(["client_id", "visit_id", "date_time"])
    .reset_index(drop=True)
)

timed_steps[["client_id", "visit_id", "process_step", "date_time", "variation"]].head(10)

,client_id,visit_id,process_step,date_time,variation
0,555,637149525_38041617439_716659,start,2017-04-15 12:57:56,Test
1,555,637149525_38041617439_716659,step_1,2017-04-15 12:58:03,Test
2,555,637149525_38041617439_716659,step_2,2017-04-15 12:58:35,Test
3,555,637149525_38041617439_716659,step_3,2017-04-15 13:00:14,Test
4,555,637149525_38041617439_716659,confirm,2017-04-15 13:00:34,Test
5,647,40369564_40101682850_311847,start,2017-04-12 15:41:28,Test
6,647,40369564_40101682850_311847,step_1,2017-04-12 15:41:35,Test
7,647,40369564_40101682850_311847,step_2,2017-04-12 15:41:53,Test
8,647,40369564_40101682850_311847,step_3,2017-04-12 15:45:02,Test
9,647,40369564_40101682850_311847,confirm,2017-04-12 15:47:45,Test


In [5]:
# We add a new column with the following timestamp so we can compare them after

timed_steps["next_step_time"] = (
    timed_steps.groupby(["client_id", "visit_id"])["date_time"].shift(-1)
)

timed_steps[["process_step", "date_time", "next_step_time", "variation"]].head(10)

,process_step,date_time,next_step_time,variation
0,start,2017-04-15 12:57:56,2017-04-15 12:58:03,Test
1,step_1,2017-04-15 12:58:03,2017-04-15 12:58:35,Test
2,step_2,2017-04-15 12:58:35,2017-04-15 13:00:14,Test
3,step_3,2017-04-15 13:00:14,2017-04-15 13:00:34,Test
4,confirm,2017-04-15 13:00:34,NaT,Test
5,start,2017-04-12 15:41:28,2017-04-12 15:41:35,Test
6,step_1,2017-04-12 15:41:35,2017-04-12 15:41:53,Test
7,step_2,2017-04-12 15:41:53,2017-04-12 15:45:02,Test
8,step_3,2017-04-12 15:45:02,2017-04-12 15:47:45,Test
9,confirm,2017-04-12 15:47:45,NaT,Test


In [6]:
# We calculate the time passed between steps and add a new column to display it

timed_steps["step_duration"] = (
    timed_steps["next_step_time"] - timed_steps["date_time"]
).dt.total_seconds()

timed_steps[["process_step", "date_time", "next_step_time", "step_duration", "variation"]].head(10)

,process_step,date_time,next_step_time,step_duration,variation
0,start,2017-04-15 12:57:56,2017-04-15 12:58:03,7.0,Test
1,step_1,2017-04-15 12:58:03,2017-04-15 12:58:35,32.0,Test
2,step_2,2017-04-15 12:58:35,2017-04-15 13:00:14,99.0,Test
3,step_3,2017-04-15 13:00:14,2017-04-15 13:00:34,20.0,Test
4,confirm,2017-04-15 13:00:34,NaT,NaN,Test
5,start,2017-04-12 15:41:28,2017-04-12 15:41:35,7.0,Test
6,step_1,2017-04-12 15:41:35,2017-04-12 15:41:53,18.0,Test
7,step_2,2017-04-12 15:41:53,2017-04-12 15:45:02,189.0,Test
8,step_3,2017-04-12 15:45:02,2017-04-12 15:47:45,163.0,Test
9,confirm,2017-04-12 15:47:45,NaT,NaN,Test


In [7]:
# We want to check how many sessions reached a specific step in the user journey
sessions_by_step = (timed_steps.drop_duplicates()
    .pivot_table(
        index="process_step",
        columns="variation",
        aggfunc="size"
    )
    .reindex(["start", "step_1", "step_2", "step_3", "confirm"])
)

display(sessions_by_step)

variation,Control,Test
process_step,,
start,45380,55773
step_1,29544,38666
step_2,25773,30899
step_3,22503,25761
confirm,17336,25600


In [8]:
# Check the median time for each step for users in group Test vs Control
median_step_times = (
    timed_steps.pivot_table(
        index="process_step",
        columns="variation",
        values="step_duration",
        aggfunc="median"
    )
    .reindex(
        index=["start", "step_1", "step_2", "step_3", "confirm"],
        columns=["Test", "Control"],
    )
)

display(median_step_times)

variation,Test,Control
process_step,,
start,14.0,20.0
step_1,27.0,20.0
step_2,61.0,64.0
step_3,58.0,72.0
confirm,93.0,46.0


In [9]:
step_duration_details = (
    timed_steps.groupby(["process_step", "variation"])["step_duration"]
    .describe()
)

display(step_duration_details.round(3))

count     mean      std  min   25%   50%     75%  \
process_step variation                                                       
confirm      Control     1920.0  168.727  410.622  0.0  13.0  46.0  129.25   
             Test        4089.0  243.690  452.196  0.0  42.0  93.0  235.00   
start        Control    35737.0   66.827  286.984  0.0   9.0  20.0   49.00   
             Test       46323.0   61.471  215.202  0.0   7.0  14.0   44.00   
step_1       Control    26043.0   50.468  153.472  0.0   9.0  20.0   47.00   
             Test       35527.0   60.675  143.254  0.0  13.0  27.0   56.00   
step_2       Control    24313.0   92.005  181.146  0.0  33.0  64.0  112.00   
             Test       29573.0   88.849  134.785  0.0  33.0  61.0  103.00   
step_3       Control    20280.0  137.135  212.662  0.0  39.0  72.0  145.00   
             Test       23983.0  129.637  223.699  0.0  31.0  58.0  127.00   

                            max  
process_step variation           
confirm      Control     9396.0  
             Test       14581.0  
start        Control    40235.0  
             Test       24819.0  
step_1       Control    10286.0  
             Test        8797.0  
step_2       Control    21763.0  
             Test       11204.0  
step_3       Control     6164.0  
             Test        6692.0

In [10]:
# how many sessions ended before reaching the step "confirm"

timed_steps[
    (timed_steps["next_step_time"].isna()) &
    (timed_steps["process_step"] != "confirm")
]["visit_id"].nunique()

32508

In [11]:
#how many sessions ended in each step

timed_steps[
    (timed_steps["next_step_time"].isna()) &
    (timed_steps["process_step"] != "confirm")
].groupby("process_step")["visit_id"].nunique()

process_step
start     19092
step_1     6640
step_2     2786
step_3     4000
Name: visit_id, dtype: int64

In [12]:
# Add a column to mention a step when it is the last one 

timed_steps["last_step"] = timed_steps["process_step"].where(
    timed_steps["next_step_time"].isna()
)

# Adding the column to the sessions_data df only when the values are not missing
sessions_data = sessions_data.merge(
    timed_steps[timed_steps["last_step"].notna()][
        ["client_id", "visit_id", "last_step"]
    ],
    on=["client_id", "visit_id"],
    how="left",
    validate="one_to_one"
)
sessions_data.head(10)

,client_id,visit_id,step_count,last_step
0,555,637149525_38041617439_716659,5,confirm
1,647,40369564_40101682850_311847,5,confirm
2,934,7076463_57954418406_971348,4,start
3,1028,557292053_87239438319_391157,9,step_1
4,1104,543158812_46395476577_767725,1,start
5,1104,643221571_99977972121_69283,1,start
6,1186,507052512_11309370126_442139,1,start
7,1186,795373564_99931517312_810896,3,step_2
8,1195,393817425_39015278493_996341,5,confirm
9,1197,71862471_21202285428_848395,7,confirm


In [13]:
# checking for mistakes in the code

timed_steps[timed_steps["client_id"] == 934]

,Unnamed: 0,client_id,visitor_id,visit_id,process_step,date_time,variation,next_step_time,step_duration,last_step
10,56,934,810392784_45004760546,7076463_57954418406_971348,start,2017-04-18 02:36:30,Test,2017-04-18 02:37:02,32.0,NaN
11,55,934,810392784_45004760546,7076463_57954418406_971348,start,2017-04-18 02:37:02,Test,2017-04-18 02:38:24,82.0,NaN
12,54,934,810392784_45004760546,7076463_57954418406_971348,start,2017-04-18 02:38:24,Test,2017-04-18 02:38:52,28.0,NaN
13,53,934,810392784_45004760546,7076463_57954418406_971348,start,2017-04-18 02:38:52,Test,NaT,NaN,start


In [14]:
#how many last_step in each step

sessions_data["last_step"].value_counts()

last_step
confirm    36927
start      19093
step_1      6640
step_3      4001
step_2      2786
Name: count, dtype: int64

In [15]:
sessions_data.head()

,client_id,visit_id,step_count,last_step
0,555,637149525_38041617439_716659,5,confirm
1,647,40369564_40101682850_311847,5,confirm
2,934,7076463_57954418406_971348,4,start
3,1028,557292053_87239438319_391157,9,step_1
4,1104,543158812_46395476577_767725,1,start


## Added on day 2 ##

In [16]:
# Code from yesterday - calculating the total session time

#Finding the start and end time of each session

session_time = web_data.groupby(["client_id", "visit_id"]).agg(start_time=("date_time", "min"),end_time=("date_time", "max")).reset_index()

session_time.head()

,client_id,visit_id,start_time,end_time
0,555,637149525_38041617439_716659,2017-04-15 12:57:56,2017-04-15 13:00:34
1,647,40369564_40101682850_311847,2017-04-12 15:41:28,2017-04-12 15:47:45
2,934,7076463_57954418406_971348,2017-04-18 02:36:30,2017-04-18 02:38:52
3,1028,557292053_87239438319_391157,2017-04-08 18:51:28,2017-04-08 19:00:26
4,1104,543158812_46395476577_767725,2017-06-12 07:49:18,2017-06-12 07:49:18


In [17]:
# Calculating the total time in seconds

session_time["total_time_seconds"] = (session_time["end_time"] - session_time["start_time"]).dt.total_seconds()

session_time.head()

,client_id,visit_id,start_time,end_time,total_time_seconds
0,555,637149525_38041617439_716659,2017-04-15 12:57:56,2017-04-15 13:00:34,158.0
1,647,40369564_40101682850_311847,2017-04-12 15:41:28,2017-04-12 15:47:45,377.0
2,934,7076463_57954418406_971348,2017-04-18 02:36:30,2017-04-18 02:38:52,142.0
3,1028,557292053_87239438319_391157,2017-04-08 18:51:28,2017-04-08 19:00:26,538.0
4,1104,543158812_46395476577_767725,2017-06-12 07:49:18,2017-06-12 07:49:18,0.0


In [18]:
# Adding the column to the sessions_data table

sessions_data = sessions_data.merge(session_time[["client_id", "visit_id", "total_time_seconds"]], on=["client_id", "visit_id"], how="left")

sessions_data.head(10)

,client_id,visit_id,step_count,last_step,total_time_seconds
0,555,637149525_38041617439_716659,5,confirm,158.0
1,647,40369564_40101682850_311847,5,confirm,377.0
2,934,7076463_57954418406_971348,4,start,142.0
3,1028,557292053_87239438319_391157,9,step_1,538.0
4,1104,543158812_46395476577_767725,1,start,0.0
5,1104,643221571_99977972121_69283,1,start,0.0
6,1186,507052512_11309370126_442139,1,start,0.0
7,1186,795373564_99931517312_810896,3,step_2,22.0
8,1195,393817425_39015278493_996341,5,confirm,245.0
9,1197,71862471_21202285428_848395,7,confirm,95.0


In [19]:
#Checking the data

print("Sessions with zero seconds:", (sessions_data["total_time_seconds"] == 0).sum())
print("\nMaximum session duration:", sessions_data["total_time_seconds"].max())

Sessions with zero seconds: 11398

Maximum session duration: 41965.0


In [20]:
## DAY 2 CODE ##

# Check only start events
starts = web_data[web_data["process_step"] == "start"]

starts.head()

,Unnamed: 0,client_id,visitor_id,visit_id,process_step,date_time,variation
4,16,555,402506806_56087378777,637149525_38041617439_716659,start,2017-04-15 12:57:56,Test
9,21,647,66758770_53988066587,40369564_40101682850_311847,start,2017-04-12 15:41:28,Test
10,53,934,810392784_45004760546,7076463_57954418406_971348,start,2017-04-18 02:38:52,Test
11,54,934,810392784_45004760546,7076463_57954418406_971348,start,2017-04-18 02:38:24,Test
12,55,934,810392784_45004760546,7076463_57954418406_971348,start,2017-04-18 02:37:02,Test


In [21]:
#Checking how many counts exist for a client in each visit 

starts_count = (starts.groupby(["client_id", "visit_id"]).size().reset_index(name="starts_count"))

starts_count.head(10)

,client_id,visit_id,starts_count
0,555,637149525_38041617439_716659,1
1,647,40369564_40101682850_311847,1
2,934,7076463_57954418406_971348,4
3,1028,557292053_87239438319_391157,1
4,1104,543158812_46395476577_767725,1
5,1104,643221571_99977972121_69283,1
6,1186,507052512_11309370126_442139,1
7,1186,795373564_99931517312_810896,1
8,1195,393817425_39015278493_996341,1
9,1197,71862471_21202285428_848395,1


In [22]:
# Adding the new column to the sessions table

sessions_data = sessions_data.merge(starts_count, on=["client_id", "visit_id"], how="left", validate="one_to_one")

sessions_data["starts_count"] = (sessions_data["starts_count"].fillna(0).astype(int))

sessions_data.head(10)

,client_id,visit_id,step_count,last_step,total_time_seconds,starts_count
0,555,637149525_38041617439_716659,5,confirm,158.0,1
1,647,40369564_40101682850_311847,5,confirm,377.0,1
2,934,7076463_57954418406_971348,4,start,142.0,4
3,1028,557292053_87239438319_391157,9,step_1,538.0,1
4,1104,543158812_46395476577_767725,1,start,0.0,1
5,1104,643221571_99977972121_69283,1,start,0.0,1
6,1186,507052512_11309370126_442139,1,start,0.0,1
7,1186,795373564_99931517312_810896,3,step_2,22.0,1
8,1195,393817425_39015278493_996341,5,confirm,245.0,1
9,1197,71862471_21202285428_848395,7,confirm,95.0,1


In [23]:
#Checking the data
print(sessions_data["starts_count"].value_counts())
print("\nMáx:",sessions_data["starts_count"].max())
print("\nMin:",sessions_data["starts_count"].min())

starts_count
1     43576
2     12174
0      5266
3      4538
4      2058
5       897
6       422
7       229
8       106
9        78
10       37
11       25
12       14
13       10
14        7
15        3
16        2
19        2
17        2
23        1
Name: count, dtype: int64

Máx: 23

Min: 0


In [24]:
# Adding new column showing entries with multiple starts

sessions_data["multiple_starts"] = (sessions_data["starts_count"] > 1)

sessions_data["multiple_starts"].value_counts()

multiple_starts
False    48842
True     20605
Name: count, dtype: int64

In [25]:
# Doing the same for confirm events

confirms = web_data[web_data["process_step"] == "confirm"]

confirms_count = (confirms.groupby(["client_id", "visit_id"]).size().reset_index(name="confirms_count"))

sessions_data = sessions_data.merge(confirms_count, on=["client_id", "visit_id"], how="left", validate="one_to_one")

sessions_data["confirms_count"] = (sessions_data["confirms_count"].fillna(0).astype(int))

In [26]:
#Checking the data
print(sessions_data["confirms_count"].value_counts())
print("\nMáx:",sessions_data["confirms_count"].max())
print("\nMin:",sessions_data["confirms_count"].min())

sessions_data["multiple_confirms"] = (sessions_data["confirms_count"] > 1)

sessions_data["multiple_confirms"].value_counts()

confirms_count
1     34333
0     31568
2      2606
3       632
4       192
5        50
8        22
6        20
7        19
9         3
10        1
11        1
Name: count, dtype: int64

Máx: 11

Min: 0


multiple_confirms
False    65901
True      3546
Name: count, dtype: int64

In [27]:
sessions_data.head(10)

,client_id,visit_id,step_count,last_step,total_time_seconds,starts_count,multiple_starts,confirms_count,multiple_confirms
0,555,637149525_38041617439_716659,5,confirm,158.0,1,False,1,False
1,647,40369564_40101682850_311847,5,confirm,377.0,1,False,1,False
2,934,7076463_57954418406_971348,4,start,142.0,4,True,0,False
3,1028,557292053_87239438319_391157,9,step_1,538.0,1,False,0,False
4,1104,543158812_46395476577_767725,1,start,0.0,1,False,0,False
5,1104,643221571_99977972121_69283,1,start,0.0,1,False,0,False
6,1186,507052512_11309370126_442139,1,start,0.0,1,False,0,False
7,1186,795373564_99931517312_810896,3,step_2,22.0,1,False,0,False
8,1195,393817425_39015278493_996341,5,confirm,245.0,1,False,1,False
9,1197,71862471_21202285428_848395,7,confirm,95.0,1,False,1,False


In [28]:
# checking sessions with multiple starts but no other step

starts_no_progress = sessions_data[
    (sessions_data["starts_count"] > 1) &
    (sessions_data["starts_count"] == sessions_data["step_count"])
]

display(starts_no_progress.head())
starts_no_progress.shape

,client_id,visit_id,step_count,last_step,total_time_seconds,starts_count,multiple_starts,confirms_count,multiple_confirms
2,934,7076463_57954418406_971348,4,start,142.0,4,True,0,False
30,2720,565429886_73206140156_730955,3,start,173.0,3,True,0,False
67,8610,243751268_9020460563_923194,2,start,18.0,2,True,0,False
77,10344,595675198_11906913204_686811,5,start,3340.0,5,True,0,False
98,15753,395225539_72187591129_857978,3,start,2478.0,3,True,0,False


(4528, 9)